In [1]:
import gymnasium as gym
import numpy as np

In [2]:
# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8


In [3]:
# --------------------------------------------------
# Policy Evaluation
# --------------------------------------------------

def policy_evaluation(policy):
    V = np.zeros(n_states)

    while True:
        delta = 0

        for s in range(n_states):
            v = V[s]
            new_v = 0

            for a, action_prob in enumerate(policy[s]):
                for prob, next_state, reward, terminated in env.P[s][a]:
                    new_v += action_prob * prob * (
                        reward + gamma * V[next_state] * (not terminated)
                    )

            V[s] = new_v
            delta = max(delta, abs(v - V[s]))

        if delta < theta:
            break

    return V

In [4]:
# --------------------------------------------------
# Policy Improvement
# --------------------------------------------------

def policy_improvement(V):
    policy = np.zeros((n_states, n_actions))

    for s in range(n_states):
        action_values = np.zeros(n_actions)

        for a in range(n_actions):
            for prob, next_state, reward, terminated in env.P[s][a]:
                action_values[a] += prob * (
                    reward + gamma * V[next_state] * (not terminated)
                )

        best_action = np.argmax(action_values)
        policy[s, best_action] = 1.0

    return policy

In [5]:
# --------------------------------------------------
# Policy Iteration
# --------------------------------------------------

policy = np.ones((n_states, n_actions)) / n_actions

while True:
    V = policy_evaluation(policy)

    new_policy = policy_improvement(V)

    if np.array_equal(policy, new_policy):
        break

    policy = new_policy

print("Optimal Value Function:")
print(V)

print("\nOptimal Policy:")
print(np.argmax(policy, axis=1))

Optimal Value Function:
[0.54202581 0.49880303 0.4706955  0.4568515  0.55845085 0.
 0.35834799 0.         0.59179866 0.64307976 0.6152075  0.
 0.         0.7417204  0.86283741 0.        ]

Optimal Policy:
[0 3 3 3 0 0 0 0 3 1 0 0 0 2 1 0]


In [7]:

# -------------------------------------------------
# Display Functions
# -------------------------------------------------

def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))


def print_policy(policy):
    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)
    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)


    print("\nOptimal Policy:")
    print(policy_grid)



In [6]:
# --------------------------------------------------
# Policy Iteration
# --------------------------------------------------

def policy_iteration(env, gamma=0.99, theta=1e-8):

    n_states = env.observation_space.n
    n_actions = env.action_space.n

    # Start with a random/equal policy
    policy = np.ones((n_states, n_actions)) / n_actions

    while True:

        # -------------------------------
        # Policy Evaluation
        # -------------------------------
        V = np.zeros(n_states)

        while True:
            delta = 0

            for s in range(n_states):

                v = V[s]
                new_v = 0

                for a in range(n_actions):

                    action_prob = policy[s][a]

                    for prob, next_state, reward, terminated in env.P[s][a]:

                        new_v += action_prob * prob * (
                            reward + gamma * V[next_state] * (not terminated)
                        )

                V[s] = new_v

                delta = max(delta, abs(v - V[s]))

            if delta < theta:
                break

        # -------------------------------
        # Policy Improvement
        # -------------------------------
        policy_stable = True

        for s in range(n_states):

            old_action = np.argmax(policy[s])

            action_values = np.zeros(n_actions)

            for a in range(n_actions):

                for prob, next_state, reward, terminated in env.P[s][a]:

                    action_values[a] += prob * (
                        reward + gamma * V[next_state] * (not terminated)
                    )

            best_action = np.argmax(action_values)

            # Update policy
            policy[s] = 0
            policy[s][best_action] = 1

            if old_action != best_action:
                policy_stable = False

        # If policy doesn't change, we're done
        if policy_stable:
            break

    return policy, V

In [8]:
# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------

optimal_policy, optimal_value_function = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("Name:vishwa v")
print("Register Number:212224110062")
print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()

Name:vishwa v
Register Number:212224110062

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Optimal Policy:
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]
